### Deliverable 1: Preprocessing the Data for a Neural Network

In [ ]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
import pandas as pd
import tensorflow as tf

#  Import and read the charity_data.csv.
import pandas as pd 
application_df = pd.read_csv("Resources/charity_data.csv")
application_df.head()

In [ ]:
# Drop the non-beneficial ID columns, 'EIN' and 'NAME'.
application_df = application_df.drop(columns = ['EIN', 'NAME'])

In [ ]:
# Determine the number of unique values in each column.
application_cat = application_df.dtypes[application_df.dtypes == "object"].index.tolist()
application_df[application_cat].nunique()

In [ ]:
# Look at APPLICATION_TYPE value counts for binning
app_type_val_counts = application_df.APPLICATION_TYPE.value_counts()
app_type_val_counts

In [ ]:
# Visualize the value counts of APPLICATION_TYPE
app_type_val_counts.plot.density()

In [ ]:
# Determine which values to replace if counts are less than ...?
replace_application = list(app_type_val_counts[app_type_val_counts < 1000].index)

# Replace in dataframe
for app in replace_application:
    application_df.APPLICATION_TYPE = application_df.APPLICATION_TYPE.replace(app,"Other")
    
# Check to make sure binning was successful
application_df.APPLICATION_TYPE.value_counts()

In [ ]:
# Look at CLASSIFICATION value counts for binning
classification_value = application_df.CLASSIFICATION.value_counts()
classification_value

In [ ]:
# Visualize the value counts of CLASSIFICATION
classification_value.plot.density()

In [ ]:
# Determine which values to replace if counts are less than ..?
replace_class = list(classification_value[classification_value < 1000].index)

# Replace in dataframe
for cls in replace_class:
    application_df.CLASSIFICATION = application_df.CLASSIFICATION.replace(cls,"Other")
    
# Check to make sure binning was successful
application_df.CLASSIFICATION.value_counts()

In [ ]:
# Look at INCOME_AMT value counts for binning
income_amt_value = application_df.INCOME_AMT.value_counts()
income_amt_value

In [ ]:
# Determine which values to replace if counts are less than ..?
replace_income = list(income_amt_value[income_amt_value < 500].index)

# Replace in dataframe
for incomes in replace_income:
    application_df.INCOME_AMT = application_df.INCOME_AMT.replace(incomes,"5M+")
    
# Check to make sure binning was successful
application_df.INCOME_AMT.value_counts()

In [ ]:
# Generate our categorical variable lists
application_cat = application_df.dtypes[application_df.dtypes == "object"].index.tolist()

application_df[application_cat].nunique()

In [ ]:
# Create a OneHotEncoder instance
enc = OneHotEncoder(sparse = False)

# Fit and transform the OneHotEncoder using the categorical variable list
encode_df = pd.DataFrame(enc.fit_transform(application_df[application_cat]))

# Add the encoded variable names to the dataframe
encode_df.columns = enc.get_feature_names(application_cat)
encode_df.head()

In [ ]:
# Merge one-hot encoded features and drop the originals
application_df = application_df.merge(encode_df, left_index = True, right_index = True)
application_df = application_df.drop(application_cat,1)
application_df.head()

In [ ]:
# Split our preprocessed data into our features and target arrays
y = application_df["IS_SUCCESSFUL"].values
X = application_df.drop(["IS_SUCCESSFUL","SPECIAL_CONSIDERATIONS_N"],1).values

# Split the preprocessed data into a training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=78)

In [ ]:
# Create a StandardScaler instances
scaler = StandardScaler()

# Fit the StandardScaler
X_scaler = scaler.fit(X_train)

# Scale the data
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

# Deliverable 2: Compile, Train and Evaluate the Model

In [ ]:
# Define the model - deep neural net, i.e., the number of input features and hidden nodes for each layer.
number_input_features = len(X_train_scaled[0])
hidden_nodes_layer1 = 90
hidden_nodes_layer2 = 60
hidden_nodes_layer3 = 40

nn=tf.keras.models.Sequential()

# First hidden layer
nn.add(tf.keras.layers.Dense(units = hidden_nodes_layer1,
                             input_dim = number_input_features,
                             activation = 'relu'))

# Second hidden layer
nn.add(tf.keras.layers.Dense(units = hidden_nodes_layer2, activation= 'relu'))

# Third hidden layer
nn.add(tf.keras.layers.Dense(units = hidden_nodes_layer3, activation = 'relu'))

# Output layer
nn.add(tf.keras.layers.Dense(units = 1, activation = "sigmoid"))

# Check the structure of the model
nn.summary()

In [ ]:
# Compile the model
nn.compile(loss = "binary_crossentropy",
           optimizer = "nadam",
           metrics = ["accuracy"])

In [ ]:
# Train the model
fit_model = nn.fit(X_train_scaled,
                   y_train,epochs = 200)

In [ ]:
# Import Dependencies
import os
from tensorflow.keras.callbacks import ModelCheckpoint

# Define the checkpoint path and filenames
os.makedirs("checkpoints_op_3/", exist_ok = True)
checkpoint_path = "checkpoints_op_3/weights.{epoch:02d}.hdf5"

In [ ]:
# Evaluate the model using the test data
model_loss, model_accuracy = nn.evaluate(X_test_scaled,y_test,verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

In [ ]:
# Compile the model
nn.compile(loss = "binary_crossentropy",
           optimizer = "nadam",
           metrics = ["accuracy"])

# Create a callback that saves the model's weights every 5 epochs
callback = ModelCheckpoint(filepath = checkpoint_path,
                              verbose = 1,
                              save_weights_only = True,
                              save_freq = 'epoch')

# Train the model
fit_model = nn.fit(X_train_scaled,
                   y_train,
                   epochs = 100,
                   callbacks = [callback])

# Evaluate the model using the test data
model_loss, model_accuracy = nn.evaluate(X_test_scaled,y_test,verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

In [ ]:
# Save and Export results to HDF5 file
nn.save('AlphabetSoupCharity_Op_03.h5')